In [1]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 5.9 MB/s eta 0:00:00


In [3]:
#!/usr/bin/env python3
"""
Complete pipeline:
1) Inputs: image_folder (settable) + CSV with columns: Image Name, Weight, View, Tomato ID
2) For each image: convert to grayscale -> Otsu threshold -> find bounding box (ground truth)
   (grayscale file temporarily saved and deleted afterwards to match requirement)
3) Split dataset into train/val/test = 80:10:10
4) Resize all images to (224,224)
5) Keep RGB pixel values, scale to [0,1], then normalize per-subset with subset mean/std
6) Random horizontal flip on training images
7) Train Faster R-CNN (ResNet-50-FPN backbone). Adam optimizer.
8) After detection, crop detected regions and save crops for next stage
9) For each crop: extract features:
   a) MobileNetV3 backbone feature (pooled scalar)
   b) Image Area (H * W)
   c) Aspect Ratio (W / H)
   d) Average Pixel Intensity (mean over pixels)
10) Input vector of 4 features -> 3-layer FC network: 4 -> 64 -> 32 -> 1 (ReLU between)
11) Loss: MSE. Optimizer: Adam.
12) Bayesian optimization (skopt) after each epoch: 5 iterations using GP surrogate and EI acquisition.
13) 3 epochs total.
14) Print MSE, MAE, R2 on test set.
15) NOTE: Otsu NOT used for inference.
"""

# -----------------------
# Install dependencies (uncomment the next lines if running in a fresh environment)
# -----------------------
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118  # or your correct torch wheel
# !pip install scikit-image opencv-python pandas scikit-learn matplotlib tqdm pillow
# !pip install scikit-optimize

import os
import shutil
import random
from pathlib import Path
import tempfile
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import box_iou

from skimage import io, color
from skimage.filters import threshold_otsu
import cv2

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Bayesian optimization
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

# -----------------------
# Config
# -----------------------
INPUT_IMAGE_FOLDER = "/content/drive/MyDrive/Tomato Yield Estimation RE/Tomato-Images/Tomato"   # <<--- edit this path to your input folder containing the images
CSV_PATH = "/content/drive/MyDrive/Tomato Yield Estimation RE/tomato.csv"      # <<--- CSV with columns: Image Name, Weight, View, Tomato ID
CROPS_DIR = "/content/drive/MyDrive/Tomato Yield Estimation RE/detected_crops"
PREPROCESSED_DIR = "/content/drive/MyDrive/Tomato Yield Estimation RE/preprocessed_images"  # resized images (224x224) for detector training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

# Detector training hyperparams (you can change)
DETECTOR_LR = 1e-4
DETECTOR_EPOCHS = 3   # keep small for demo; change as necessary
DETECTOR_BATCH_SIZE = 1  # detection usually uses 1

# Weight estimation MLP training
MLP_EPOCHS = 3 # as requested
MLP_BATCH_SIZE = 32

# Bayesian optimization config (per epoch)
BO_N_CALLS = 5 # as requested
BO_INITIAL_POINTS = 2

IMG_SIZE = (224, 224)
RANDOM_FLIP_PROB = 0.5

# Ensure reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Create directories
os.makedirs(CROPS_DIR, exist_ok=True)
os.makedirs(PREPROCESSED_DIR, exist_ok=True)

# -----------------------
# Helper: Otsu-based bbox extraction
# -----------------------
def otsu_bbox_from_image(img_path, tmp_gray_path=None):
    """
    1) Convert to grayscale
    2) Save grayscale image temporarily if tmp_gray_path provided
    3) Use Otsu thresholding to create mask
    4) Find largest contour and return bbox (x, y, w, h)
    5) Delete tmp grayscale file if it was saved
    """
    img = io.imread(img_path)
    if img.ndim == 3:
        gray = color.rgb2gray(img)  # values in [0,1]
        gray_uint8 = (gray * 255).astype(np.uint8)
    else:
        gray_uint8 = (img * 255).astype(np.uint8) if img.max() <= 1 else img.astype(np.uint8)

    # optionally save grayscale to disk and then delete
    if tmp_gray_path:
        cv2.imwrite(tmp_gray_path, gray_uint8)

    # Otsu threshold
    try:
        th = threshold_otsu(gray_uint8)
    except Exception:
        th = 128
    _, mask = cv2.threshold(gray_uint8, int(th), 255, cv2.THRESH_BINARY)

    # find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        h, w = gray_uint8.shape[:2]
        return (0, 0, w, h)  # fallback full image

    # pick largest contour by area
    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)

    # delete grayscale file if saved
    if tmp_gray_path and os.path.exists(tmp_gray_path):
        os.remove(tmp_gray_path)

    return (x, y, w, h)

# -----------------------
# Step 1: Read CSV and build dataset list
# -----------------------
df = pd.read_csv(CSV_PATH)
required_cols = ['Image Name', 'Weight', 'View', 'Tomato ID']
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"CSV must contain column: {c}")

# Build a list of samples
samples = []
for _, row in df.iterrows():
    img_name = row['Image Name']
    weight = float(row['Weight'])
    img_path = os.path.join(INPUT_IMAGE_FOLDER, img_name)
    if not os.path.exists(img_path):
        print(f"Warning: {img_path} not found — skipping")
        continue
    samples.append({"img_path": img_path, "weight": weight, "view": row['View'], "id": row['Tomato ID']})

if len(samples) == 0:
    raise RuntimeError("No images found. Check INPUT_IMAGE_FOLDER and CSV_PATH.")

# -----------------------
# Step 2: Use Otsu thresholding to generate ground-truth bboxes
#   (grayscale temp files are deleted after)
# -----------------------
print("Generating ground-truth boxes via Otsu thresholding (grayscale saved then removed)...")
gt_records = []  # each record: img_path, weight, bbox(x,y,w,h)
tmpdir = tempfile.mkdtemp(prefix="tmp_gray_")
for s in tqdm(samples):
    img_path = s["img_path"]
    tmp_gray = os.path.join(tmpdir, Path(img_path).stem + "_gray.png")
    bbox = otsu_bbox_from_image(img_path, tmp_gray_path=tmp_gray)
    gt_records.append({"img_path": img_path, "weight": s["weight"], "bbox": bbox, "view": s["view"], "id": s["id"]})

# Remove the tmpdir (all grayscale images removed by function, but ensure)
try:
    shutil.rmtree(tmpdir)
except Exception:
    pass

# -----------------------
# Step 3: Split into train/val/test (80:10:10)
# -----------------------
paths = [r["img_path"] for r in gt_records]
weights = [r["weight"] for r in gt_records]
bbs = [r["bbox"] for r in gt_records]

train_idx, temp_idx = train_test_split(range(len(paths)), test_size=0.2, random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED)  # split 0.2 into 0.1 and 0.1

splits = {'train': train_idx, 'val': val_idx, 'test': test_idx}
print(f"Dataset sizes -> train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")

# -----------------------
# Step 4 & 5: Resize all images to (224,224) and save into PREPROCESSED_DIR / subset folders
# Also store the resized RGB arrays for normalization step
# -----------------------
print("Resizing images to 224x224 and writing preprocessed copies...")
subset_images = {'train': [], 'val': [], 'test': []}  # each item: dict with keys img (np array RGB float [0,1]), bbox (rescaled), weight
for split_name, idxs in splits.items():
    out_dir = os.path.join(PREPROCESSED_DIR, split_name)
    os.makedirs(out_dir, exist_ok=True)
    for i in idxs:
        rec = gt_records[i]
        img = io.imread(rec["img_path"])
        # Ensure RGB
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        if img.shape[2] == 4:
            img = img[..., :3]
        # original bbox coordinates were on original image. We'll compute scaled bbox after resize.
        orig_h, orig_w = img.shape[:2]
        x, y, w, h = rec["bbox"]
        # Resize
        resized = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
        # compute scale
        scale_x = IMG_SIZE[1] / orig_w
        scale_y = IMG_SIZE[0] / orig_h
        bx = int(round(x * scale_x))
        by = int(round(y * scale_y))
        bw = int(round(w * scale_x))
        bh = int(round(h * scale_y))
        # clamp
        bx = max(0, min(bx, IMG_SIZE[1] - 1))
        by = max(0, min(by, IMG_SIZE[0] - 1))
        bw = max(1, min(bw, IMG_SIZE[1] - bx))
        bh = max(1, min(bh, IMG_SIZE[0] - by))
        # save resized image to disk (RGB)
        out_path = os.path.join(out_dir, Path(rec["img_path"]).name)
        io.imsave(out_path, resized.astype(np.uint8))
        subset_images[split_name].append({
            "img_path": out_path,
            "img_array": (resized.astype(np.float32) / 255.0),  # scaled to [0,1] right away
            "bbox": (bx, by, bw, bh),
            "weight": rec["weight"]
        })
print("Resizing complete.")

# -----------------------
# Step 6 & 7 done: at this point images are RGB and scaled to [0,1]
# Step 8: compute per-subset mean/std and normalize
# -----------------------
print("Computing subset-wise mean/std and applying normalization...")
subset_meanstd = {}
for split_name in ['train', 'val', 'test']:
    arrays = [d['img_array'] for d in subset_images[split_name]]
    if len(arrays) == 0:
        raise RuntimeError(f"No images in split {split_name}")
    # stack channel wise
    stacked = np.stack(arrays, axis=0)  # (N, H, W, C)
    # compute per-channel mean/std
    # change to (N, C, H, W) for channel axis
    stacked_ch = np.transpose(stacked, (0, 3, 1, 2))
    mean = stacked_ch.mean(axis=(0, 2, 3))
    std = stacked_ch.std(axis=(0, 2, 3))
    subset_meanstd[split_name] = {'mean': mean, 'std': std}
    # normalize each stored img_array
    for d in subset_images[split_name]:
        # (H, W, C) -> (C, H, W)
        img_ch = np.transpose(d['img_array'], (2, 0, 1))
        img_ch = (img_ch - mean[:, None, None]) / (std[:, None, None] + 1e-9)
        d['img_array_norm'] = img_ch  # normalized tensor-like np array

print("Normalization stats per subset:")
for s, v in subset_meanstd.items():
    print(f"{s}: mean={v['mean']}, std={v['std']}")

# -----------------------
# Step 9: Random horizontal flip for training set will be applied on-the-fly in dataset
# -----------------------

# -----------------------
# Create custom Dataset for Faster R-CNN training
# Each item returns: PIL image (Tensor) and target dict with boxes and labels
# -----------------------
class DetectionDataset(Dataset):
    def __init__(self, records, transforms=None, normalize_mean=None, normalize_std=None, apply_flip=False):
        """
        records: list of dicts with keys img_path, bbox (x,y,w,h), weight (unused here)
        transforms: torchvision transforms to apply to the image (expecting normalized numpy arrays)
        normalize_mean/std not used because we already normalized per-subset; but keep placeholder
        apply_flip: whether to randomly horizontally flip (for training)
        """
        self.records = records
        self.apply_flip = apply_flip

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = io.imread(r['img_path'])  # uint8 HWC
        img = img.astype(np.float32) / 255.0  # [0,1]
        # convert to tensor CHW
        img_t = torch.from_numpy(np.transpose(img, (2,0,1))).float()
        # targets: boxes in [x1,y1,x2,y2], labels -> single class 1 (food)
        x, y, w, h = r['bbox']
        boxes = torch.tensor([[x, y, x + w, y + h]], dtype=torch.float32)
        labels = torch.tensor([1], dtype=torch.int64)  # we set 1 as "food" (single-class detection)
        target = {"boxes": boxes, "labels": labels}
        # optionally apply random horizontal flip on both image and box
        if self.apply_flip and random.random() < RANDOM_FLIP_PROB:
            img_t = torch.flip(img_t, dims=[2])  # horizontal flip along width dimension
            _, H, W = img_t.shape
            # flip box
            x1, y1, x2, y2 = boxes[0]
            new_x1 = W - x2
            new_x2 = W - x1
            boxes = torch.tensor([[new_x1, y1, new_x2, y2]], dtype=torch.float32)
            target["boxes"] = boxes
        return img_t, target

def collate_fn(batch):
    imgs = [b[0] for b in batch]
    targets = [b[1] for b in batch]
    return imgs, targets

# Build detection datasets and dataloaders (train uses flip)
det_train_ds = DetectionDataset(subset_images['train'], apply_flip=True)
det_val_ds = DetectionDataset(subset_images['val'], apply_flip=False)
det_test_ds = DetectionDataset(subset_images['test'], apply_flip=False)

det_train_loader = DataLoader(det_train_ds, batch_size=DETECTOR_BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
det_val_loader = DataLoader(det_val_ds, batch_size=DETECTOR_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
det_test_loader = DataLoader(det_test_ds, batch_size=DETECTOR_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# -----------------------
# Step 10: Faster R-CNN with ResNet backbone initialization
# Single-class detection (food vs background)
# -----------------------
print("Initializing Faster R-CNN (ResNet-50-FPN) ...")
num_classes = 2  # background + food
detector = fasterrcnn_resnet50_fpn(pretrained=True)
# replace the head classifier to match num_classes
in_features = detector.roi_heads.box_predictor.cls_score.in_features
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
detector.to(DEVICE)

optimizer_det = optim.Adam(detector.parameters(), lr=DETECTOR_LR)

# training loop for detector
print("Training Faster R-CNN detector...")
detector.train()
for epoch in range(DETECTOR_EPOCHS):
    epoch_loss = 0.0
    for images, targets in tqdm(det_train_loader, desc=f"Detector Epoch {epoch+1}/{DETECTOR_EPOCHS}"):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = detector(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        optimizer_det.zero_grad()
        losses.backward()
        optimizer_det.step()
        epoch_loss += losses.item()
    print(f"Detector epoch {epoch+1} loss: {epoch_loss:.4f}")

print("Detector training complete.")

# -----------------------
# Step 11: Use detector to create crops (detected bounding boxes)
# Save crops to CROPS_DIR/{split}/
# -----------------------
print("Running detector on all splits to generate detected crops...")
detector.eval()
with torch.no_grad():
    for split_name in ['train', 'val', 'test']:
        split_out = os.path.join(CROPS_DIR, split_name)
        os.makedirs(split_out, exist_ok=True)
        records = subset_images[split_name]
        for idx, rec in enumerate(tqdm(records, desc=f"Detect & crop {split_name}")):
            img = io.imread(rec['img_path']).astype(np.float32) / 255.0
            img_t = torch.from_numpy(np.transpose(img, (2,0,1))).float().to(DEVICE)
            outputs = detector([img_t])
            # outputs[0]['boxes'] [N,4], scores [N]
            boxes = outputs[0].get('boxes', torch.empty((0,4))).cpu().numpy()
            scores = outputs[0].get('scores', torch.empty((0,))).cpu().numpy() # <<< FIX: Move scores to CPU and convert to numpy
            if len(boxes) == 0:
                # fallback to ground-truth bbox
                x,y,w,h = rec['bbox']
                boxes = np.array([[x,y,x+w,y+h]], dtype=np.int32)
                chosen_box = boxes[0]
            else:
                # choose highest-score box
                best_idx = int(np.argmax(scores))
                chosen_box = boxes[best_idx].astype(int)
            x1,y1,x2,y2 = chosen_box
            # clamp
            H, W = img.shape[:2]
            x1, y1, x2, y2 = max(0, x1), max(0,y1), min(W-1,x2), min(H-1,y2)
            crop = img[y1:y2+1, x1:x2+1, :]
            # Save cropped image for feature extraction
            out_path = os.path.join(split_out, f"{Path(rec['img_path']).stem}_crop.png")
            io.imsave(out_path, (crop * 255).astype(np.uint8))
            rec.setdefault('detected_crop_path', out_path)
print("Crops generated and saved.")

# -----------------------
# Step 12: Feature extraction per crop using MobileNetV3 backbone
# We'll extract a feature vector per crop:
#   a) backbone scalar alpha = global pooled feature mean
#   b) Image area = H*W
#   c) Aspect ratio = W/H
#   d) Average pixel intensity = mean of pixels (grayscale)
# -----------------------
print("Preparing MobileNetV3 backbone for feature extraction...")
mobilenet_weights = MobileNet_V3_Large_Weights.IMAGENET1K_V2
mobilenet = mobilenet_v3_large(weights=mobilenet_weights).to(DEVICE)
mobilenet.eval()
# remove the classifier head: keep features and adaptive pooling
# mobilenet.features -> feature map
# final pooling uses mobilenet.classifier normally; we'll use AdaptiveAvgPool2d manually
pool = nn.AdaptiveAvgPool2d((1,1)).to(DEVICE)

def extract_features_from_crop(crop_path):
    img = io.imread(crop_path).astype(np.float32) / 255.0  # HWC [0,1]
    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)
    if img.shape[2] == 4:
        img = img[..., :3]
    # ensure tensor shape expected by mobilenet (C,H,W), normalized with mobilenet's weights transforms
    # apply mobilenet preprocessing
    preprocess = mobilenet_weights.transforms()
    pil = (img * 255).astype(np.uint8)
    # use PIL image via skimage -> convert to uint8 array then to PIL
    from PIL import Image
    pil_img = Image.fromarray(pil)
    inp = preprocess(pil_img).unsqueeze(0).to(DEVICE)  # shape (1,3,H,W)
    with torch.no_grad():
        feat_map = mobilenet.features(inp)  # shape (1, C, h, w)
        pooled = pool(feat_map)  # (1, C, 1, 1)
        vec = pooled.view(1, -1)  # (1, C)
        # to scalar alpha, take mean of feature vector
        alpha = float(vec.mean().cpu().numpy())
    H, W = img.shape[:2]
    area = float(H * W)
    aspect_ratio = float(W / H) if H != 0 else 0.0
    avg_pixel_intensity = float(img.mean())
    return alpha, area, aspect_ratio, avg_pixel_intensity

# Build feature datasets
print("Extracting features for all crops (this might take a while)...")
features_dict = {'train': [], 'val': [], 'test': []}  # each entry: (feature_vector [4], weight)
for split_name in ['train', 'val', 'test']:
    for rec in tqdm(subset_images[split_name], desc=f"Feature extract {split_name}"):
        crop_path = rec.get('detected_crop_path')
        if not crop_path or not os.path.exists(crop_path):
            # fallback to original resized image crop using bbox
            img = io.imread(rec['img_path']).astype(np.float32) / 255.0
            x,y,w,h = rec['bbox']
            crop = img[y:y+h, x:x+w, :]
            tmp_path = os.path.join(CROPS_DIR, "tmp_crop.png")
            io.imsave(tmp_path, (crop * 255).astype(np.uint8))
            crop_path = tmp_path
        alpha, area, ar, api = extract_features_from_crop(crop_path)
        features_dict[split_name].append((np.array([alpha, area, ar, api], dtype=np.float32), rec['weight']))

print("Feature extraction complete.")

# -----------------------
# Step 13: Now train the 3-layer FC regression model using features
#   Input: vector (4,) ; Layers: 4 -> 64 (ReLU) -> 32 (ReLU) -> 1
# -----------------------
class WeightDataset(Dataset):
    def __init__(self, feature_pairs):
        # feature_pairs: list of tuples (np.array(4,), target float)
        self.data = feature_pairs

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

train_ds = WeightDataset(features_dict['train'])
val_ds = WeightDataset(features_dict['val'])
test_ds = WeightDataset(features_dict['test'])

train_loader = DataLoader(train_ds, batch_size=MLP_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=MLP_BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=MLP_BATCH_SIZE, shuffle=False)

# define model
class WeightMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x.squeeze(-1)

# instantiate
mlp = WeightMLP().to(DEVICE)
criterion = nn.MSELoss()

# -----------------------
# Step 16: Bayesian optimization setup (we'll tune learning_rate and weight_decay for the MLP optimizer)
# We'll run a short BO (n_calls=BO_N_CALLS) after each epoch using validation MSE as objective.
# -----------------------
space = [
    Real(1e-6, 1e-2, "log-uniform", name="lr"),
    Real(1e-8, 1e-2, "log-uniform", name="weight_decay")
]

def train_one_epoch(model, optimizer, loader):
    model.train()
    total_loss = 0.0
    for X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
    return total_loss / len(loader.dataset)

def eval_mse(model, loader):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(DEVICE)
            out = model(X)
            preds.append(out.cpu().numpy())
            trues.append(y.numpy())
    preds = np.concatenate(preds).ravel()
    trues = np.concatenate(trues).ravel()
    return mean_squared_error(trues, preds), preds, trues

# Bayesian objective (we retrain MLP quickly on train set with given hyperparams and evaluate on val set)
def bo_objective_factory(train_loader, val_loader):
    @use_named_args(space)
    def objective(**params):
        lr = params["lr"]
        wd = params["weight_decay"]
        # instantiate a fresh small model and train for 1 epoch (cheap surrogate)
        temp_model = WeightMLP().to(DEVICE)
        opt = optim.Adam(temp_model.parameters(), lr=lr, weight_decay=wd)
        # train for 1 epoch on train set (fast)
        train_one_epoch(temp_model, opt, train_loader)
        val_mse, _, _ = eval_mse(temp_model, val_loader)
        # objective to minimize
        return val_mse
    return objective

# Top-level MLP training with BO after each epoch
best_bo_params_per_epoch = []
print("Training MLP with Bayesian optimization tuning after each epoch...")
for epoch in range(MLP_EPOCHS):
    # initial MLP optimizer (start with default lr)
    optimizer = optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
    # Train one epoch
    train_loss = train_one_epoch(mlp, optimizer, train_loader)
    val_mse, _, _ = eval_mse(mlp, val_loader)
    print(f"Epoch {epoch+1}/{MLP_EPOCHS} initial train_loss={train_loss:.6f} val_mse={val_mse:.6f}")

    # Run Bayesian optimization (GP + Expected Improvement) to find better hyperparams
    print(f"Running Bayesian optimization (n_calls={BO_N_CALLS}) to tune lr & weight_decay (using val set)...")
    bo_obj = bo_objective_factory(train_loader, val_loader)
    res = gp_minimize(bo_obj, dimensions=space, n_calls=BO_N_CALLS, n_initial_points=BO_INITIAL_POINTS, acq_func="EI", random_state=SEED)
    best_params = {'lr': float(res.x[0]), 'weight_decay': float(res.x[1])}
    best_bo_params_per_epoch.append(best_params)
    print(f"Best BO params this epoch: {best_params}, best val_mse: {res.fun:.6f}")

    # Optionally, continue training main mlp for another epoch with found params (we'll update optimizer)
    optimizer = optim.Adam(mlp.parameters(), lr=best_params['lr'], weight_decay=best_params['weight_decay'])
    train_loss2 = train_one_epoch(mlp, optimizer, train_loader)
    val_mse2, _, _ = eval_mse(mlp, val_loader)
    print(f"Epoch {epoch+1} after BO train_loss={train_loss2:.6f} val_mse={val_mse2:.6f}")

print("MLP training complete.")

# -----------------------
# Step 17: Evaluate on test set and print MSE, MAE, R^2
# -----------------------
test_mse, preds, trues = eval_mse(mlp, test_loader)
test_mae = mean_absolute_error(trues, preds)
test_r2 = r2_score(trues, preds)
print(f"Test results -> MSE: {test_mse:.6f}, MAE: {test_mae:.6f}, R^2: {test_r2:.6f}")

# Print BO results per epoch
for i, bp in enumerate(best_bo_params_per_epoch):
    print(f"Epoch {i+1} best BO params: {bp}")

# -----------------------
# Save final models and scalers
# -----------------------
MODEL_OUT_DIR = "final_models"
os.makedirs(MODEL_OUT_DIR, exist_ok=True)
torch.save(detector.state_dict(), os.path.join(MODEL_OUT_DIR, "fasterrcnn_resnet50_fpn_detector.pth"))
torch.save(mlp.state_dict(), os.path.join(MODEL_OUT_DIR, "weight_mlp.pth"))
# Save normalization info
import json
norm_info = {k: {'mean': v['mean'].tolist(), 'std': v['std'].tolist()} for k, v in subset_meanstd.items()}
with open(os.path.join(MODEL_OUT_DIR, "normalization_info.json"), "w") as f:
    json.dump(norm_info, f, indent=2)

print("Saved detector, MLP and normalization info in", MODEL_OUT_DIR)

# -----------------------
# Important: Inference note
# Use detector to find boxes in a new image (do NOT use Otsu during inference), crop detected regions,
# compute features via MobileNetV3, then pass features through MLP to predict weight.
# -----------------------
def infer_weight(image_path, detector_model, mlp_model):
    detector_model.eval()
    mlp_model.eval()
    # read & preprocess
    img = io.imread(image_path).astype(np.float32) / 255.0
    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)
    if img.shape[2] == 4:
        img = img[..., :3]
    # resize to IMG_SIZE and run detector
    resized = cv2.resize((img * 255).astype(np.uint8), IMG_SIZE)  # keep same preprocessing as training resizing
    inp = torch.from_numpy(np.transpose(resized.astype(np.float32)/255.0, (2,0,1))).float().to(DEVICE)
    with torch.no_grad():
        outs = detector_model([inp])
    boxes = outs[0].get('boxes', torch.empty((0,4))).cpu().numpy()
    scores = outs[0].get('scores', torch.empty((0,))).cpu().numpy() # <<< FIX: Move scores to CPU and convert to numpy
    if len(boxes) == 0:
        # fallback whole image
        crop = resized.astype(np.float32) / 255.0
    else:
        best_idx = int(np.argmax(scores))
        b = boxes[best_idx].astype(int)
        x1,y1,x2,y2 = b
        H, W = resized.shape[:2]
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W-1, x2), min(H-1, y2)
        crop = resized[y1:y2+1, x1:x2+1, :].astype(np.float32) / 255.0
    # extract features
    tmp_crop_path = os.path.join(tempfile.gettempdir(), "infer_tmp_crop.png")
    io.imsave(tmp_crop_path, (crop * 255).astype(np.uint8))
    alpha, area, ar, api = extract_features_from_crop(tmp_crop_path)
    os.remove(tmp_crop_path)
    feat = torch.tensor([alpha, area, ar, api], dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = mlp_model(feat).cpu().item()
    return pred

# demonstration: infer on a single test image (if exists)
if len(subset_images['test']) > 0:
    demo_img = subset_images['test'][0]['img_path']
    pred_weight = infer_weight(demo_img, detector, mlp)
    print(f"Demo inference on {demo_img}: predicted weight = {pred_weight}, true = {subset_images['test'][0]['weight']}")

print("Pipeline finished successfully.")

Generating ground-truth boxes via Otsu thresholding (grayscale saved then removed)...


100%|██████████| 45/45 [00:32<00:00,  1.37it/s]


Dataset sizes -> train: 36, val: 4, test: 5
Resizing images to 224x224 and writing preprocessed copies...
Resizing complete.
Computing subset-wise mean/std and applying normalization...
Normalization stats per subset:
train: mean=[0.65927655 0.6382598  0.57298046], std=[0.14956197 0.17207319 0.21779206]
val: mean=[0.6632596 0.6438142 0.5803643], std=[0.15236935 0.17254308 0.21623473]
test: mean=[0.65845925 0.6370155  0.5715316 ], std=[0.1555507  0.17637478 0.22417778]
Initializing Faster R-CNN (ResNet-50-FPN) ...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Training Faster R-CNN detector...


...

Crops generated and saved.
Preparing MobileNetV3 backbone for feature extraction...
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 153MB/s]


Extracting features for all crops (this might take a while)...


Feature extract test: 100%|██████████| 5/5 [00:00<00:00, 76.44it/s]


Feature extraction complete.
Training MLP with Bayesian optimization tuning after each epoch...
Epoch 1/3 initial train_loss=1762612.638889 val_mse=22995.771484
Running Bayesian optimization (n_calls=5) to tune lr & weight_decay (using val set)...
Best BO params this epoch: {'lr': 1e-06, 'weight_decay': 0.000734732148815563}, best val_mse: 813076.500000
Epoch 1 after BO train_loss=22564.869792 val_mse=22651.517578
Epoch 2/3 initial train_loss=40285.628472 val_mse=19964.914062
Running Bayesian optimization (n_calls=5) to tune lr & weight_decay (using val set)...
Best BO params this epoch: {'lr': 1e-06, 'weight_decay': 9.345185500703693e-06}, best val_mse: 23657.318359
Epoch 2 after BO train_loss=22172.280382 val_mse=19643.035156
Epoch 3/3 initial train_loss=39928.098958 val_mse=22954.564453
Running Bayesian optimization (n_calls=5) to tune lr & weight_decay (using val set)...
Best BO params this epoch: {'lr': 0.0013145103232150136, 'weight_decay': 3.811544088653074e-05}, best val_mse: 2